In [ ]:
#Model Building and Evaluation(MILESTONE 3)

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)


In [4]:
# Load cleaned dataset
df = pd.read_csv("cleaned_dataset.csv")

print(df.shape)
df.head()


(9997, 19)


,order_id,supplier_id,supplier_rating,supplier_lead_time,order_date,promised_delivery_date,actual_delivery_date,shipment_mode,shipping_distance_km,order_quantity,unit_price,total_order_value,weather_condition,region,holiday_period,previous_on_time_rate,carrier_name,delayed_reason_code,on_time_delivery
0,4,9832,3.9,7,2024-08-12,2024-08-19,2024-08-19,Air,839,71,365.79,25971.09,Rainy,India,No,85.2,FedEx,Operational,1
1,5,2126,3.2,8,2024-07-07,2024-07-15,2024-07-18,Sea,258,9,3052.84,27475.56,Cloudy,USA,No,77.1,BlueDart,Customs,0
2,6,4401,3.1,8,2024-05-05,2024-05-13,2024-05-14,Road,450,19,2569.78,4552.40,Cloudy,North,Yes,79.3,DHL,Traffic,0
3,7,8725,4.0,10,2024-10-14,2024-10-24,2024-10-25,Air,278,98,448.95,43997.10,Cloudy,South,Yes,87.2,Delhivery,Traffic,0
4,8,4079,4.2,7,2024-03-10,2024-03-17,2024-03-16,Sea,176,50,2833.97,141698.50,Rainy,East,No,92.2,FedEx,Operational,1


In [5]:
TARGET = "on_time_delivery"

X = df.drop(columns=[TARGET])
y = df[TARGET]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))


Train class distribution:
on_time_delivery
0    0.719645
1    0.280355
Name: proportion, dtype: float64

Test class distribution:
on_time_delivery
0    0.7195
1    0.2805
Name: proportion, dtype: float64


In [7]:
#Correct ways to handle a Date column

In [7]:
date_cols = ['order_date', 'promised_delivery_date', 'actual_delivery_date']

existing_date_cols = [col for col in date_cols if col in df.columns]
print("Date columns found:", existing_date_cols)

for col in existing_date_cols:
    df[col] = pd.to_datetime(df[col])


Date columns found: ['order_date', 'promised_delivery_date', 'actual_delivery_date']


In [8]:
if 'actual_delivery_date' in df.columns and 'promised_delivery_date' in df.columns:
    df['delivery_delay_days'] = (
        df['actual_delivery_date'] - df['promised_delivery_date']
    ).dt.days


In [9]:
df.drop(columns=existing_date_cols, inplace=True)


In [10]:
X = df.drop('on_time_delivery', axis=1)
y = df['on_time_delivery']


In [11]:
numeric_features = [
    'supplier_id',
    'supplier_rating',
    'supplier_lead_time',
    'shipping_distance_km',
    'order_quantity',
    'unit_price',
    'total_order_value',
    'previous_on_time_rate',
    'delivery_delay_days'
]

categorical_features = [
    'shipment_mode',
    'weather_condition',
    'region',
    'holiday_period',
    'carrier_name',
    'delayed_reason_code'
]


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [14]:
log_model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))


Logistic Regression Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1439
           1       1.00      1.00      1.00       561

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [32]:
#: Confirm imbalance

In [15]:
# Check class distribution
print("Class counts:")
print(y.value_counts())

print("\nClass percentages:")
print(y.value_counts(normalize=True) * 100)


Class counts:
on_time_delivery
0    7194
1    2803
Name: count, dtype: int64

Class percentages:
on_time_delivery
0    71.961588
1    28.038412
Name: proportion, dtype: float64


In [34]:
#checking feature columns

In [16]:
print(X.columns)


Index(['order_id', 'supplier_id', 'supplier_rating', 'supplier_lead_time',
       'shipment_mode', 'shipping_distance_km', 'order_quantity', 'unit_price',
       'total_order_value', 'weather_condition', 'region', 'holiday_period',
       'previous_on_time_rate', 'carrier_name', 'delayed_reason_code',
       'delivery_delay_days'],
      dtype='object')


In [17]:
X = df.drop(columns=[
    'on_time_delivery',
    'delivery_delay_days',
    'delayed_reason_code'
])

y = df['on_time_delivery']


In [ ]:
#model1(log reg)

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Feature types
numeric_features = X.select_dtypes(include=['int64','float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Preprocessor
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Pipeline
log_model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(
        max_iter=3000,
        class_weight='balanced',
        random_state=42
    ))
])

# Train
log_model.fit(X_train, y_train)

# Evaluate
y_pred = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.5215
              precision    recall  f1-score   support

           0       0.73      0.53      0.61      1439
           1       0.29      0.50      0.37       561

    accuracy                           0.52      2000
   macro avg       0.51      0.52      0.49      2000
weighted avg       0.61      0.52      0.55      2000



In [39]:
#Random forest

In [40]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.7015
              precision    recall  f1-score   support

           0       0.72      0.96      0.82      1439
           1       0.28      0.04      0.07       561

    accuracy                           0.70      2000
   macro avg       0.50      0.50      0.45      2000
weighted avg       0.60      0.70      0.61      2000



In [41]:
#XGBoost

In [42]:
pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [43]:
from xgboost import XGBClassifier

xgb_model = Pipeline([
    ('prep', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=7194/2803,  # class imbalance handling
        eval_metric='logloss',
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
 

XGBoost Accuracy: 0.6035
              precision    recall  f1-score   support

           0       0.72      0.73      0.73      1439
           1       0.29      0.29      0.29       561

    accuracy                           0.60      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.60      0.60      0.60      2000



In [48]:
#Again train the XGBoost

In [51]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    scale_pos_weight=7194/2803
)

param_grid = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth': [4, 6, 8],
    'clf__learning_rate': [0.03, 0.05, 0.1],
    'clf__subsample': [0.7, 0.8],
    'clf__colsample_bytree': [0.7, 0.8]
}

xgb_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', xgb)
])

grid_xgb = GridSearchCV(
    xgb_pipeline,
    param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_xgb.fit(X_train, y_train)

print("Best Params:", grid_xgb.best_params_)


Fitting 3 folds for each of 72 candidates, totalling 216 fits
Best Params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.03, 'clf__max_depth': 4, 'clf__n_estimators': 200, 'clf__subsample': 0.8}


In [53]:
best_xgb = grid_xgb.best_estimator_

y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


XGBoost Accuracy: 0.536
XGBoost ROC-AUC: 0.5012368710198085
              precision    recall  f1-score   support

           0       0.73      0.57      0.64      1439
           1       0.29      0.45      0.35       561

    accuracy                           0.54      2000
   macro avg       0.51      0.51      0.49      2000
weighted avg       0.60      0.54      0.56      2000



In [54]:
#Handle class imbalance

In [56]:
!pip install imbalanced-learn


In [62]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [63]:


# 1. Define features


In [64]:
numeric_features = ['supplier_rating','supplier_lead_time','shipping_distance_km',
                    'order_quantity','unit_price','total_order_value','previous_on_time_rate','delivery_delay_days']
categorical_features = ['shipment_mode','weather_condition','region','holiday_period','carrier_name','delayed_reason_code']

X = df[numeric_features + categorical_features]
y = df['on_time_delivery']


In [65]:
# 2. Train-test split

In [66]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [67]:
# 3. Column Transformer

In [68]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [69]:
# 4. Full Pipeline with SMOTE

In [75]:
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=42))
])


In [71]:
# 5. Train model

In [76]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [73]:
# 6. Predictions & evaluation

In [77]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

XGBoost Accuracy: 1.0
XGBoost ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1439
           1       1.00      1.00      1.00       561

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [78]:
#data leakage happend again!

In [79]:
#Remove any leakage features

In [81]:
cols_to_drop = ['on_time_delivery', 'delivery_delay_days', 'actual_delivery_date', 'promised_delivery_date']
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
y = df['on_time_delivery']


In [82]:
#Do train-test split:

In [83]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [84]:
#Build your pipeline without leakage features

In [86]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier

numeric_features = ['supplier_rating', 'supplier_lead_time', 'shipping_distance_km', 'order_quantity', 'unit_price', 'total_order_value', 'previous_on_time_rate']
categorical_features = ['shipment_mode','weather_condition','region','holiday_period','carrier_name','delayed_reason_code']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(eval_metric='logloss',  random_state=42))
])

pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [87]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


Accuracy: 0.676
ROC-AUC: 0.5172350575203863
              precision    recall  f1-score   support

           0       0.72      0.90      0.80      1439
           1       0.27      0.09      0.14       561

    accuracy                           0.68      2000
   macro avg       0.50      0.50      0.47      2000
weighted avg       0.59      0.68      0.61      2000



In [90]:
#Use a ColumnTransformer in a pipeline

In [94]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

# ====== 1. Define features ======
# Check your actual columns in df
print(df.columns)

# Numeric features (only columns that exist)
numeric_features = [
    'supplier_rating', 
    'supplier_lead_time', 
    'shipping_distance_km', 
    'order_quantity',
    'unit_price', 
    'total_order_value', 
    'previous_on_time_rate'
]

# Categorical features (only columns that exist)
categorical_features = [
    'shipment_mode',
    'weather_condition',
    'region',
    'holiday_period',
    'carrier_name',
    'delayed_reason_code'
]

# ====== 2. Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# ====== 3. Split data ======
X = df[numeric_features + categorical_features]
y = df['on_time_delivery']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ====== 4. Create pipeline ======
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(random_state=42))
])

# ====== 5. GridSearchCV ======
param_grid = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 5, 10],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

# ====== 6. Fit model ======
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

# ====== 7. Evaluate ======
y_pred = grid_search.predict(X_test)
y_prob = grid_search.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Index(['order_id', 'supplier_id', 'supplier_rating', 'supplier_lead_time',
       'shipment_mode', 'shipping_distance_km', 'order_quantity', 'unit_price',
       'total_order_value', 'weather_condition', 'region', 'holiday_period',
       'previous_on_time_rate', 'carrier_name', 'delayed_reason_code',
       'on_time_delivery', 'delivery_delay_days'],
      dtype='object')
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best parameters: {'rf__max_depth': None, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 100}
Accuracy: 0.7185
F1-Score: 0.01054481546572935
Precision: 0.375
Recall: 0.0053475935828877
ROC-AUC: 0.47953247885799083
Confusion Matrix:
 [[1434    5]
 [ 558    3]]

Classification Report:
               precision    recall  f1-score   support

           0       0.72      1.00      0.84      1439
           1       0.38      0.01      0.01       561

    accuracy                           0.72      2000
   macro avg       0.55      0.50